# nbimplot Public Examples (Complete)

This notebook is a release-style showcase for `nbimplot`, covering all supported plot types,
interaction tools, axis controls, subplots, aligned layouts, and callback hooks.

All examples are Jupyter-native, WASM + ImPlot backed.


## Table of Contents

1. Setup
2. Core Line Workflows (`line`, `set_data`, `stream_line`, `append`)
3. Realtime Streaming Updates
4. Scatter Family (`scatter`, `bubbles`)
5. Step / Stem / Digital (`stairs`, `stems`, `digital`)
6. Bars Family (`bars`, `bars_h`, `bar_groups`)
7. Area + Uncertainty (`shaded`, `error_bars`, `error_bars_h`)
8. Reference Lines (`inf_lines`, `vlines`, `hlines`)
9. Distribution (`histogram`, `histogram2d`)
10. Grid and Raster (`heatmap`, `image`)
11. Pie Charts (`pie_chart`)
12. Textual Primitives (`text`, `annotation`, `dummy`)
13. Axis Tags and Colormap Widgets (`tag_x`, `tag_y`, `colormap_*`)
14. Drag Tools (`drag_line_x`, `drag_line_y`, `drag_point`, `drag_rect`)
15. Drag/Drop Targets (`drag_drop_plot`, `drag_drop_axis`, `drag_drop_legend`)
16. Axis and View Controls
17. Explicit Scale / Autoscale / Manual Aligned Group
18. Callback Hooks (`on_tool_change`, `on_selection_change`, `on_perf_stats`)
19. Subplots (`Subplots`, `set_subplots_config`, `AlignedPlots`)
20. Rich Composite Example


In [16]:
import numpy as np
import nbimplot as ip
from IPython.display import display

rng = np.random.default_rng(2026)


def smooth_noise(n, scale=1.0):
    x = rng.normal(0.0, scale, int(n)).astype(np.float32)
    k = np.array([0.1, 0.2, 0.4, 0.2, 0.1], dtype=np.float32)
    return np.convolve(x, k, mode="same").astype(np.float32)


def demo_signal(n, drift=0.0, amp=1.0):
    x = np.linspace(0.0, 20.0, int(n), dtype=np.float32)
    y = (
        amp * np.sin(1.7 * x)
        + 0.35 * np.cos(0.35 * x + 0.5)
        + 0.15 * np.sin(5.5 * x)
        + drift * (x / x.max())
        + 0.08 * smooth_noise(n, 1.0)
    ).astype(np.float32)
    return x, y


## 1) Core Line Workflows

Includes:
- multi-line plotting
- styling (`color`, `line_weight`, `marker`, `marker_size`)
- update path (`set_data`, `render`)
- streaming path (`stream_line`, `append`)
- hidden-item utility (`hide_next_item`)


In [2]:
x, y = demo_signal(40_000)
_, y2 = demo_signal(40_000, drift=0.4, amp=0.7)

p = ip.Plot(width=1100, height=420, title="Line: Style + Update")
h1 = p.line("mid", y, x_axis="x1", y_axis="y1", color="#1d4ed8", line_weight=1.8)
h2 = p.line("vwap", y2, color="#ea580c", line_weight=1.4, marker="none")

# Demonstrate hidden next item (e.g. internal guide)
p.hide_next_item()
p.line("hidden-guide", np.zeros_like(y), color="#71717a")

p.show()

# Update existing series in-place and redraw
h2.set_data((y2 * 1.05 + 0.08).astype(np.float32))
p.render()


In [3]:
base = np.linspace(0, 1, 512, dtype=np.float32)
stream = ip.Plot(width=1100, height=300, title="Streaming Line: Fixed Capacity Ring Buffer")
sh = stream.stream_line("stream", capacity=4000, initial=base)
stream.show()

for k in range(8):
    chunk = (np.sin(np.linspace(0, 1.4, 640, dtype=np.float32) + k * 0.45) + 0.03 * smooth_noise(640)).astype(np.float32)
    sh.append(chunk)  # keeps only latest capacity points
stream.render()


## 1b) Realtime Streaming Updates

A timed update loop using `stream_line` + `append` + `render`.

Run this cell to see live updates for ~10 seconds. Stop early with the notebook interrupt button.


In [ ]:
import time

p = ip.Plot(width=1150, height=320, title="Realtime Streaming (Timed Loop)")
h = p.stream_line("rt", capacity=60_000, initial=np.zeros(2000, dtype=np.float32))
p.show()

t = 0.0
for frame in range(200):  # ~10 seconds at 20 FPS
    chunk_x = np.linspace(t, t + 0.10, 500, dtype=np.float32)
    chunk = (
        np.sin(10.0 * chunk_x)
        + 0.35 * np.sin(3.3 * chunk_x + 0.6)
        + 0.06 * rng.normal(size=chunk_x.size)
    ).astype(np.float32)
    h.append(chunk)
    p.render()
    t += 0.10
    time.sleep(0.05)


## 2) Scatter Family

Includes:
- `scatter`
- `bubbles`


In [4]:
n = 2600
x = np.linspace(-5, 5, n, dtype=np.float32)
y = (0.25 * x**2 + 0.8 * np.sin(2.2 * x) + 0.25 * smooth_noise(n)).astype(np.float32)
sz = (6.0 + 18.0 * (np.abs(np.sin(x)))).astype(np.float32)

p = ip.Plot(width=1100, height=360, title="Scatter + Bubbles")
p.scatter("scatter", y, x=x, size=2.0)
p.bubbles("bubbles", y * 0.82, sz, x=x)
p.show()


## 3) Step / Stem / Digital

Includes:
- `stairs`
- `stems`
- `digital`


In [5]:
n = 200
x = np.arange(n, dtype=np.float32)
y_step = (np.cumsum(rng.normal(0, 0.2, n)).astype(np.float32))
y_stem = (np.abs(np.sin(np.linspace(0, 8, n))).astype(np.float32))
y_dig = ((np.sin(np.linspace(0, 40, n)) > 0).astype(np.float32))

p = ip.Plot(width=1100, height=360, title="Stairs / Stems / Digital")
p.stairs("stairs", y_step, x=x)
p.stems("stems", y_stem, x=x)
p.digital("digital", y_dig, x=x)
p.show()


## 4) Bars Family

Includes:
- `bars`
- `bars_h`
- `bar_groups`


In [6]:
x = np.arange(12, dtype=np.float32)
a = (2.0 + np.abs(np.sin(x / 1.8) * 4.0)).astype(np.float32)
b = (1.5 + np.abs(np.cos(x / 2.4) * 3.5)).astype(np.float32)

p = ip.Plot(width=1100, height=380, title="Bars / Horizontal Bars / Bar Groups")
p.bars("bars", a, x=x, bar_width=0.55)
p.bars_h("bars_h", b, y=x, bar_height=0.45)
p.bar_groups(["desk-a", "desk-b"], np.vstack([a, b]), group_size=0.72, shift=0.08)
p.show()


## 5) Area + Uncertainty

Includes:
- `shaded`
- `error_bars` (symmetric + asymmetric)
- `error_bars_h` (asymmetric)


In [7]:
x = np.linspace(0, 30, 900, dtype=np.float32)
y_mid = (np.sin(x * 0.4) + 0.1 * smooth_noise(x.size)).astype(np.float32)
y_lo = (y_mid - 0.35 - 0.05 * np.cos(x)).astype(np.float32)
y_hi = (y_mid + 0.35 + 0.05 * np.sin(2 * x)).astype(np.float32)

pts_x = np.linspace(1, 29, 20, dtype=np.float32)
pts_y = (np.sin(pts_x * 0.4) + 0.05 * smooth_noise(pts_x.size)).astype(np.float32)
err = (0.12 + 0.05 * np.abs(np.sin(pts_x))).astype(np.float32)
err_neg = (0.06 + 0.04 * np.abs(np.cos(pts_x))).astype(np.float32)
err_pos = (0.12 + 0.06 * np.abs(np.sin(pts_x * 0.7))).astype(np.float32)

p = ip.Plot(width=1100, height=420, title="Shaded + Error Bars")
p.shaded("band", y_lo, y_hi, x=x, alpha=0.25)
p.line("mid", y_mid, color="#0369a1", line_weight=1.8)
p.error_bars("sym", pts_y, err, x=pts_x)
p.error_bars("asym", pts_y + 0.2, err_neg=err_neg, err_pos=err_pos, x=pts_x)
p.error_bars_h("asym_h", x=pts_y + 2.5, y=pts_x, err_neg=err_neg, err_pos=err_pos)
p.show()


## 6) Reference Lines

Includes:
- `inf_lines`
- `vlines`
- `hlines`


In [8]:
_, y = demo_signal(4000)

p = ip.Plot(width=1100, height=320, title="Infinite Reference Lines")
p.line("signal", y, color="#0f766e")
p.inf_lines("inf-x", np.array([300, 1200, 2500], dtype=np.float32), axis="x")
p.vlines("vlines", np.array([700, 1800, 3200], dtype=np.float32))
p.hlines("hlines", np.array([-0.8, 0.0, 0.8], dtype=np.float32))
p.show()


## 7) Distribution Plots

Includes:
- `histogram`
- `histogram2d` with colorbar controls


In [17]:
s = rng.normal(loc=0.2, scale=1.3, size=80_000).astype(np.float32)
x = rng.normal(loc=0.0, scale=1.1, size=120_000).astype(np.float32)
y = (0.55 * x + rng.normal(0, 0.8, size=x.size)).astype(np.float32)

p = ip.Plot(width=1100, height=380, title="Histogram + Histogram2D")
p.histogram("hist", s, bins=80)
p.histogram2d(
    "hist2d",
    x,
    y,
    x_bins=96,
    y_bins=72,
    label_fmt="",
    show_colorbar=True,
    colorbar_label="Count",
    colorbar_format="%g",
)
p.show()


## 8) Grid and Raster

Includes:
- `heatmap` (label formatting + colorbar)
- `image` with bounds, UV, tint, and flags


In [18]:
rows, cols = 120, 160
xx = np.linspace(-2.5, 2.5, cols, dtype=np.float32)
yy = np.linspace(-2.0, 2.0, rows, dtype=np.float32)
X, Y = np.meshgrid(xx, yy)
Z = (np.sin(2.1 * X) * np.cos(2.6 * Y) + 0.2 * np.exp(-(X**2 + Y**2))).astype(np.float32)

p = ip.Plot(width=1100, height=420, title="Heatmap")
p.heatmap(
    "heat",
    Z,
    label_fmt="",
    show_colorbar=True,
    colorbar_label="Intensity",
    colorbar_format="%.2f",
)
p.show()


In [19]:
h, w = 180, 260
xg = np.linspace(0, 1, w, dtype=np.float32)
yg = np.linspace(0, 1, h, dtype=np.float32)
X, Y = np.meshgrid(xg, yg)
img_rgb = np.stack([
    (0.5 + 0.5 * np.sin(7 * X)).astype(np.float32),
    (0.5 + 0.5 * np.cos(5 * Y)).astype(np.float32),
    (0.4 + 0.6 * np.sin(3 * X + 2 * Y)).astype(np.float32),
], axis=-1)

p = ip.Plot(width=1100, height=420, title="Image (RGB + Bounds + Tint)")
p.image(
    "rgb",
    img_rgb,
    bounds=((10.0, -4.0), (40.0, 6.0)),
    uv0=(0.0, 0.0),
    uv1=(1.0, 1.0),
    tint=(1.0, 1.0, 1.0, 0.92),
    image_flags=0,
)
p.show()


## 9) Pie Chart

Includes:
- `pie_chart` with labels, radius, and formatting


In [20]:
vals = np.array([32, 21, 17, 14, 9, 7], dtype=np.float32)
labels = ["rates", "equities", "credit", "fx", "commod", "other"]

p = ip.Plot(width=800, height=420, title="Pie Chart")
p.pie_chart("alloc", vals, labels=labels, x=0.0, y=0.0, radius=1.25, angle0=120.0, label_fmt="%.1f")
p.show()


## 10) Textual Primitives

Includes:
- `text`
- `annotation`
- `dummy`


In [21]:
x = np.linspace(0, 18, 1000, dtype=np.float32)
y = (np.sin(x) + 0.1 * np.cos(5 * x)).astype(np.float32)

p = ip.Plot(width=1100, height=340, title="Text + Annotation + Dummy")
p.line("curve", y)
p.text("start", float(x[40]), float(y[40]))
p.annotation("local max", float(x[260]), float(y[260]), offset_x=10, offset_y=-18)
p.dummy("legend-note")
p.show()


## 11) Axis Tags + Colormap Widgets

Includes:
- `tag_x`, `tag_y`
- `colormap_slider`, `colormap_button`, `colormap_selector`
- `set_colormap`


In [22]:
x, y = demo_signal(2600)

p = ip.Plot(width=1100, height=360, title="Tags + Colormap Widgets")
p.set_colormap("Viridis")
p.line("signal", y)
p.tag_x(6.0, label_fmt="x=%.2f", round_value=False)
p.tag_y(0.0, label_fmt="y=%.2f", round_value=True)
p.colormap_slider(label="Blend", t=0.45, fmt="%.2f")
p.colormap_button(label="Cycle", width=80, height=18)
p.colormap_selector(label="Colormap")
p.show()


## 12) Drag Tools

Includes:
- `drag_line_x`
- `drag_line_y`
- `drag_point`
- `drag_rect`


In [23]:
x, y = demo_signal(2500)

p = ip.Plot(width=1100, height=360, title="Drag Tools")
p.line("signal", y, color="#0f766e")
p.drag_line_x("entry-x", 5.0, thickness=1.4)
p.drag_line_y("entry-y", 0.2, thickness=1.4)
p.drag_point("pivot", 7.0, -0.5, size=6.0)
p.drag_rect("zone", 8.0, -1.0, 11.0, 1.1)
p.show()


## 13) Drag/Drop Targets

Includes:
- `drag_drop_plot`
- `drag_drop_axis`
- `drag_drop_legend`

Use mouse drag/drop inside the plot and legend to trigger events.


In [24]:
x, y = demo_signal(3000)

p = ip.Plot(width=1100, height=360, title="Drag/Drop Targets")
p.line("signal-a", y, color="#1d4ed8")
p.line("signal-b", (0.7 * y + 0.3).astype(np.float32), color="#ea580c")
p.drag_drop_plot(source=True, target=True)
p.drag_drop_axis("x1", source=True, target=True)
p.drag_drop_axis("y1", source=True, target=True)
p.drag_drop_legend(target=True)
p.show()


## 14) Axis and View Controls

Includes:
- scales, labels, formats, custom ticks
- axis constraints and links
- secondary axes and time axis
- plot flags
- view controls (`set_view`, `autoscale`)


In [25]:
p = ip.Plot(width=1180, height=430, title="Axis Controls + View")
p.set_plot_flags(no_menus=False, no_box_select=False, crosshairs=True)
p.set_secondary_axes(x2=True, y2=True)

x = np.linspace(0, 200, 6000, dtype=np.float32)
y1 = (0.5 * np.sin(x / 8) + 0.02 * smooth_noise(x.size)).astype(np.float32)
y2 = (80 + 40 * np.cos(x / 23)).astype(np.float32)

p.line("primary", y1, x_axis="x1", y_axis="y1", color="#0369a1")
p.line("secondary", y2, x_axis="x2", y_axis="y2", color="#b91c1c")

p.set_axis_label("x1", "Samples")
p.set_axis_label("y1", "Amplitude")
p.set_axis_label("x2", "Samples (linked)")
p.set_axis_label("y2", "Spread")

p.set_axis_format("y1", "%.3f")
p.set_axis_format("y2", "%.1f")

custom_ticks = np.array([0, 40, 80, 120, 160, 200], dtype=np.float32)
p.set_axis_ticks("x1", custom_ticks, labels=[str(v) for v in custom_ticks.astype(int)], keep_default=False)

p.set_axis_limits_constraints("x1", 0.0, 200.0)
p.set_axis_zoom_constraints("x1", 1.0, 80.0)
p.set_axis_link("x2", "x1")

p.set_view(20.0, 120.0, -1.2, 1.2)
p.show()


In [26]:
# Clear custom ticks and switch to time formatting on x-axis (epoch seconds)

p2 = ip.Plot(width=1180, height=360, title="Time Axis")
base = 1_762_000_000.0  # epoch seconds
xt = (base + np.arange(2400, dtype=np.float32) * 60.0).astype(np.float32)
yt = (100 + np.cumsum(0.05 * rng.normal(size=xt.size)).astype(np.float32)).astype(np.float32)

p2.line("close", yt)
p2.set_time_axis("x1")
p2.set_axis_label("x1", "Time")
p2.set_axis_label("y1", "Price")
p2.show()

# Example API usage for clearing ticks explicitly:
# p.clear_axis_ticks("x1")


## 14b) Explicit Scale / Autoscale / Manual Aligned Group

Direct usage examples for:
- `set_axis_scale`
- `autoscale`
- `set_aligned_group`


In [27]:
p = ip.Plot(width=1100, height=360, title="Scale + Autoscale + Aligned Group")
p.set_axis_scale(x="log", y="linear")
p.set_aligned_group("manual-group", enabled=True, vertical=True)

x = np.logspace(0, 4, 4000, dtype=np.float32)
y = (0.3 * np.log10(x) + 0.2 * np.sin(np.log(x))).astype(np.float32)
p.line("log-x series", y, color="#0f766e")

p.autoscale()
p.show()


## 15) Callback Hooks

Includes:
- `on_tool_change`
- `on_selection_change`
- `on_perf_stats`

Interact with the plot (drag tools / box-select) and inspect captured event buffers.


In [28]:
tool_events = []
selection_events = []
perf_events = []


def on_tool(_plot, payload):
    tool_events.append(payload)


def on_select(_plot, payload):
    selection_events.append(payload)


def on_perf(_plot, payload):
    perf_events.append(payload)


x, y = demo_signal(6000)
p = ip.Plot(width=1180, height=380, title="Callbacks + Interaction")
p.line("signal", y)
p.drag_line_x("dlx", 3.5)
p.drag_rect("box", 7.0, -0.7, 10.0, 0.9)

p.on_tool_change(on_tool)
p.on_selection_change(on_select)
p.on_perf_stats(on_perf, interval_ms=300)

p.show()

print("Interact with the plot, then inspect: tool_events, selection_events, perf_events")


Interact with the plot, then inspect: tool_events, selection_events, perf_events


## 16) Native Subplots (`ip.Subplots`)

Clean, high-level subplot API with per-cell plotting methods.


In [29]:
sp = ip.Subplots(
    2,
    2,
    title="Subplots API",
    width=1200,
    height=780,
    link_rows=True,
    link_cols=True,
    share_items=True,
)

x1, y1 = demo_signal(5000)
x2, y2 = demo_signal(2800, drift=0.2)
sp.subplot(0, 0).line("line", y1)
sp.subplot(0, 1).scatter("scatter", y2, x=x2)
sp.subplot(1, 0).histogram("hist", rng.normal(size=50_000).astype(np.float32), bins=60)
sp.subplot(1, 1).heatmap("hm", (rng.normal(size=(80, 120))).astype(np.float32), label_fmt="")
sp.show()


## 17) Direct Subplots via `Plot.set_subplots_config`

Low-level direct control with `subplot_index` on each primitive/series.


In [30]:
p = ip.Plot(width=1200, height=780, title="Direct Subplot Config")
p.set_subplots_config(rows=2, cols=2, flags=0)

x, y = demo_signal(4200)
p.line("s0", y, subplot_index=0)
p.bars("s1", np.abs(np.sin(np.linspace(0, 8, 30))).astype(np.float32), subplot_index=1)
p.histogram2d(
    "s2",
    rng.normal(size=45_000).astype(np.float32),
    rng.normal(size=45_000).astype(np.float32),
    x_bins=64,
    y_bins=64,
    label_fmt="",
    subplot_index=2,
)
p.pie_chart("s3", np.array([5, 3, 2, 1], dtype=np.float32), labels=["A", "B", "C", "D"], subplot_index=3)
p.show()


## 18) Aligned Plots (`ip.AlignedPlots`)

Stacked aligned layout for easier cross-panel comparisons.


In [31]:
al = ip.AlignedPlots(2, 1, group_id="release-aligned", vertical=True, width=1200, height=700, title="Aligned Plots")

x, y = demo_signal(6000)
al.subplot(0, 0).line("price", y, color="#1d4ed8")
al.subplot(0, 0).vlines("events", np.array([1000, 2600, 4200], dtype=np.float32))

vol = (0.3 + np.abs(np.sin(np.linspace(0, 18, x.size)))).astype(np.float32)
al.subplot(1, 0).bars("volume", vol, x=np.arange(vol.size, dtype=np.float32), bar_width=0.8)
al.show()


## 19) Rich Composite Example (Everything Together)

Single dashboard-style example mixing many primitives and controls for public-demo quality.


In [32]:
dash = ip.Plot(width=1280, height=820, title="Composite Demo")
dash.set_subplots_config(rows=3, cols=2, flags=0)
dash.set_colormap("Plasma")

# Panel 0: lines + tags + drag tools
x, y = demo_signal(7000)
dash.line("signal", y, subplot_index=0, color="#0f766e", line_weight=1.7)
dash.tag_x(8.0, subplot_index=0, label_fmt="x=%.2f")
dash.tag_y(0.0, subplot_index=0, label_fmt="y=%.2f")
dash.drag_line_x("dx", 6.0, subplot_index=0)
dash.drag_point("dp", 10.0, 0.4, subplot_index=0)

# Panel 1: scatter + bubbles
xs = np.linspace(-4, 4, 1800, dtype=np.float32)
ys = (0.6 * np.sin(2.3 * xs) + 0.22 * smooth_noise(xs.size)).astype(np.float32)
bs = (5 + 16 * np.abs(np.cos(xs))).astype(np.float32)
dash.scatter("sc", ys, x=xs, subplot_index=1)
dash.bubbles("bb", ys * 0.7, bs, x=xs, subplot_index=1)

# Panel 2: shaded + error bars
x3 = np.linspace(0, 16, 800, dtype=np.float32)
y3 = np.sin(x3).astype(np.float32)
dash.shaded("band", y3 - 0.25, y3 + 0.25, x=x3, subplot_index=2)
dash.error_bars("err", y3[::40], 0.08 * np.ones_like(y3[::40]), x=x3[::40], subplot_index=2)

# Panel 3: histogram2d + heatmap overlay style
x4 = rng.normal(size=80_000).astype(np.float32)
y4 = (0.4 * x4 + rng.normal(scale=0.9, size=x4.size)).astype(np.float32)
dash.histogram2d("h2d", x4, y4, x_bins=90, y_bins=70, label_fmt="", show_colorbar=True, subplot_index=3)

# Panel 4: image
h, w = 100, 140
u = np.linspace(0, 1, w, dtype=np.float32)
v = np.linspace(0, 1, h, dtype=np.float32)
U, V = np.meshgrid(u, v)
img = np.stack([U, V, 0.5 + 0.5 * np.sin(7 * U * V)], axis=-1).astype(np.float32)
dash.image("img", img, subplot_index=4, bounds=((0.0, 0.0), (20.0, 10.0)))

# Panel 5: pie + text/annotation
vals = np.array([30, 24, 18, 16, 12], dtype=np.float32)
dash.pie_chart("pie", vals, labels=["A", "B", "C", "D", "E"], subplot_index=5)
dash.text("mix", 0.0, 0.0, subplot_index=5)
dash.annotation("allocation", 0.0, 0.0, subplot_index=5)

# Global drag/drop + colormap widgets
for i in range(6):
    dash.drag_drop_plot(subplot_index=i)

dash.colormap_selector(label="Map", subplot_index=0)
dash.colormap_slider(label="T", t=0.5, subplot_index=0)
dash.colormap_button(label="Next", width=72, height=18, subplot_index=0)

dash.show()


## Notes

- For very large time-series, prefer `line` + view interaction + built-in WASM LOD.
- For streaming, prefer `stream_line(..., capacity=...)` + `append(...)`.
- For synchronized layouts, use `Subplots` or `AlignedPlots`.
- For advanced integrations, register callbacks and keep event handlers lightweight.
